In [57]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import time
import csv
import pandas as pd

In [58]:
# Khởi tạo trình duyệt 
options = webdriver.ChromeOptions()
options.add_experimental_option("detach", True)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [59]:
# Danh sách các năm cần thu thập từ menu bên trái
years = range(2011, 2027)
rated_links = []

# Mở tệp tin để ghi kết quả 
with open("moveek_rated_movie_links.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    
    # Ghi tiêu đề cột (Header)
    writer.writerow(["Review Link"]) 
    
    for year in years:
        url = f"https://moveek.com/phim-viet-nam-{year}/"
        driver.get(url)
        time.sleep(2) 
         # Tìm tất cả các thành phần phim có class là 'item'
        movie_items = driver.find_elements(By.CLASS_NAME, "item")
        
        for movie in movie_items:
            try:
                # Kiểm tra xem phim có nhãn đánh giá hài lòng (class 'text-success') hay không
                rating_check = movie.find_elements(By.CLASS_NAME, "text-success")
                
                if len(rating_check) > 0:
                    # Trích xuất thẻ <a> có đường dẫn bắt đầu bằng '/review/'
                    link_element = movie.find_element(By.CSS_SELECTOR, "a[href^='/review/']")
                    movie_link = link_element.get_attribute("href")
                    
                    if movie_link not in rated_links:
                        rated_links.append(movie_link)
                        # Ghi dòng dữ liệu vào file CSV
                        writer.writerow([movie_link])
                        
            except Exception as e:
                continue


In [60]:
movie_links = []
try:
    with open("moveek_rated_movie_links.csv", "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        next(reader)  # Bỏ qua tiêu đề
        for row in reader:
            if row: movie_links.append(row[0])
    print(f"Bắt đầu thu thập thông tin cho {len(movie_links)} bộ phim...")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file rated_movie_links.csv!")

# --- PHẦN 2: THU THẬP THÔNG TIN CHI TIẾT PHIM ---
with open("moveek_movie_details.csv", "w", newline="", encoding="utf-8-sig") as f_out:
    writer = csv.writer(f_out)
    # Định dạng file CSV kết quả
    writer.writerow(["Movie Title", "Genres", "Year", "Country", "Duration", "Link"])

    for link in movie_links:
        try:
            print(f"Đang lấy thông tin: {link}")
            driver.get(link)
            time.sleep(3) # Đợi trang tải đầy đủ

            # 1. Lấy tên phim 
            try:
                title = driver.find_element(By.CSS_SELECTOR, "h1.text-truncate a").text
            except:
                title = "N/A"

            # 2. Lấy thông tin chung (Năm, Thể loại)
            try:
                # 1. Tìm thẻ p chứa cả tên tiếng Anh và thể loại
                raw_meta = driver.find_element(By.CSS_SELECTOR, ".mb-3.text-center p.text-muted").text
    
                # Ví dụ raw_meta: " The Ultimate Of Love - Comedy, Drama, Family "
                if "-" in raw_meta:
                # 2. Cắt chuỗi tại dấu "-" và lấy phần phía sau
                    genres = raw_meta.split("-")[-1].strip()
                else:
                # Nếu không có dấu "-" thì lấy hết hoặc để N/A
                    genres = raw_meta.strip()
        
            except Exception as e:
                genres = "N/A"

            try:
                year = driver.find_element(By.XPATH, "//i[contains(@class, 'fe-calendar')]/ancestor::div[1]/span").text.strip()
            except:
                year = "N/A"

            try:
                duration = driver.find_element(By.XPATH, "//i[contains(@class, 'fe fe-clock')]/ancestor::div[1]/span[last()]").text.strip()
            except:
                duration = "N/A"

            # Ghi vào file CSV
            writer.writerow([title, genres, year, "Việt Nam", duration, link])
            
        except Exception as e:
            print(f"Không thể lấy thông tin phim tại {link}: {e}")
            continue

print("Đã hoàn thành thu thập hồ sơ phim!")

Bắt đầu thu thập thông tin cho 109 bộ phim...
Đang lấy thông tin: https://moveek.com/review/teo-em/
Đang lấy thông tin: https://moveek.com/review/qua-tim-mau-2013/
Đang lấy thông tin: https://moveek.com/review/scandal-hao-quang-tro-lai/
Đang lấy thông tin: https://moveek.com/review/de-mai-tinh-2/
Đang lấy thông tin: https://moveek.com/review/chung-cu-ma/
Đang lấy thông tin: https://moveek.com/review/chang-trai-nam-ay/
Đang lấy thông tin: https://moveek.com/review/ngay-nay-ngay-nay/
Đang lấy thông tin: https://moveek.com/review/love-yeu/
Đang lấy thông tin: https://moveek.com/review/em-la-ba-noi-cua-anh/
Đang lấy thông tin: https://moveek.com/review/sieu-trom/
Đang lấy thông tin: https://moveek.com/review/loc-phat/
Đang lấy thông tin: https://moveek.com/review/gai-gia-lam-chieu/
Đang lấy thông tin: https://moveek.com/review/taxi-em-ten-gi/
Đang lấy thông tin: https://moveek.com/review/vo-oi-em-o-dau/
Đang lấy thông tin: https://moveek.com/review/truy-sat/
Đang lấy thông tin: https://mov

In [61]:
# 1. Đọc danh sách link từ file CSV đã lưu
movie_links1 = []
try:
    with open("moveek_rated_movie_links.csv", "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        next(reader)  # Bỏ qua dòng tiêu đề (Review Link)
        for row in reader:
            if row:  # Kiểm tra dòng không rỗng
                movie_links1.append(row[0])
except FileNotFoundError:
    print("File not found rated_movie_links.csv!")

# 2. Mở file CSV mới để lưu BÌNH LUẬN
with open("moveek_movie_reviews_data.csv", "w", newline="", encoding="utf-8-sig") as f_out:
    writer = csv.writer(f_out)
    # Ghi tiêu đề cho file kết quả
    writer.writerow(["Reviewer Name", "Review Content", "Rating Score", "Movie Link",])

    for link in movie_links1:
        try:
            driver.get(link)
            time.sleep(3) # Chờ bình luận tải
            
            while True:
                try:
                    # Tìm nút "Xem thêm" 
                    load_more_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Xem thêm')]")
                
                    # Cuộn xuống để nút hiển thị trong tầm mắt (tránh lỗi click chặn)
                    driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
                    time.sleep(1)
                
                    load_more_button.click()
                    time.sleep(2) # Chờ bình luận mới load ra
                except (NoSuchElementException, ElementClickInterceptedException):
                    # Không tìm thấy nút nữa hoặc nút bị che (đã load hết)
                    break
                except Exception as e:
                    print(f"Event failed: {e}")
                    break
            # Tìm danh sách các khối bình luận
            review_items = driver.find_elements(By.CLASS_NAME, "card-body") # Ví dụ class name
            
            for item in review_items:
                try:
                    # Trích xuất thông tin chi tiết
                    name = item.find_element(By.CSS_SELECTOR, ".card-title a").text
                    content = item.find_element(By.CLASS_NAME, "review-content").text
                    score = item.find_element(By.CSS_SELECTOR, ".card-title span").text.strip()
                    
                    # Ghi trực tiếp vào file CSV kết quả
                    writer.writerow([name, content, score, link])
                except:
                    continue
            
        except Exception as e:
            continue